In [ ]:
npy_path='/content/images_mls.npy'
csv_path='/content/Labels_mls.csv'

In [ ]:
KaggleNNCVDataset(npy_path, csv_path)

In [ ]:
initialize_kaggle_loaders(npy_path, csv_path)

--- Data Encoding Report ---
Successfully cataloged 10 monkey categories.


(<torch.utils.data.dataloader.DataLoader at 0x791519575be0>,
 <torch.utils.data.dataloader.DataLoader at 0x7915195769c0>)

In [ ]:
npy_path='/content/images_mls.npy'
csv_path='/content/Labels_mls.csv'

In [ ]:
# =====================================================================
# CELL 1: DEPENDENCIES & CORE INITIALIZATION
# =====================================================================
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split

# Detect GPU or fall back to CPU seamlessly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Execution Engine initialized on compute target: {device}")

# =====================================================================
# CELL 2: PROGRAMMATIC WORKSPACE DIRECTORY SETUP
# =====================================================================
folders = ['Data', 'Models', 'Outputs']
print("📁 Structuring clean project directories...")
for folder in folders:
    if not os.path.exists(folder):
        os.makedirs(folder)
        print(f"  └─ Created sub-directory: /{folder}")

# Programmatically organize raw source files into /Data if left in root
dataset_migrations = {
    'Labels_mls.csv': 'Data/Labels_mls.csv',
    'images_mls.npy': 'Data/images_mls.npy'
}
for src, dest in dataset_migrations.items():
    if os.path.exists(src):
        shutil.move(src, dest)
        print(f"➡️ Consolidated pipeline asset: '{src}' shifted to /{dest}")

# Set pipeline paths targeting your organized Data folder structure
npy_path = 'Data/images_mls.npy'
csv_path = 'Data/Labels_mls.csv'

# =====================================================================
# CELL 3: DATA ENGINE ARCHITECTURE (DATASET & DATALOADER)
# =====================================================================
class KaggleNNCVDataset(Dataset):
    def __init__(self, images_array, labels_vector, transform=None):
        self.images = images_array
        self.labels = labels_vector
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        label = self.labels[idx]

        if self.transform:
            img = self.transform(img)
        else:
            img = np.transpose(img, (2, 0, 1))
            img = torch.tensor(img, dtype=torch.float32) / 255.0

        return img, torch.tensor(label, dtype=torch.long)

def initialize_kaggle_loaders(npy_path, csv_path):
    raw_images = np.load(npy_path)
    labels_df = pd.read_csv(csv_path)

    string_labels = labels_df.iloc[:, 0].values
    unique_species = sorted(list(np.unique(string_labels)))
    label_to_id = {species_name: idx for idx, species_name in enumerate(unique_species)}

    print("\n--- Data Encoding Report ---")
    print(f"Successfully cataloged {len(unique_species)} monkey categories.")

    numerical_labels = np.array([label_to_id[name] for name in string_labels])

    X_train, X_val, y_train, y_val = train_test_split(
        raw_images, numerical_labels, test_size=0.2, random_state=812, stratify=numerical_labels
    )

    train_transforms = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((128, 128)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    val_transforms = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = KaggleNNCVDataset(X_train, y_train, transform=train_transforms)
    val_dataset = KaggleNNCVDataset(X_val, y_val, transform=val_transforms)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)

    return train_loader, val_loader

# FIX: Active declaration assigning loaders to real runtime variables!
train_loader, val_loader = initialize_kaggle_loaders(npy_path, csv_path)

# =====================================================================
# CELL 4: MODEL CONFIGURATION AND CUSTOM HEAD DECLARATION
# =====================================================================
class TunableMonkeyEfficientNet(nn.Module):
    def __init__(self, num_classes=10, config=None):
        super(TunableMonkeyEfficientNet, self).__init__()

        if config is None:
            config = {
                'dense_neurons': 512,
                'dropout_1': 0.4,
                'dropout_2': 0.3,
                'use_batchnorm': True
            }

        weights = models.EfficientNet_B0_Weights.DEFAULT
        self.backbone = models.efficientnet_b0(weights=weights)

        # Freeze backbones to isolate and protect feature layers
        for param in self.backbone.features.parameters():
            param.requires_grad = False

        in_features = self.backbone.classifier[1].in_features

        classifier_modules = []
        classifier_modules.append(nn.Dropout(p=config['dropout_1']))
        classifier_modules.append(nn.Linear(in_features, config['dense_neurons']))

        if config['use_batchnorm']:
            classifier_modules.append(nn.BatchNorm1d(config['dense_neurons']))

        classifier_modules.append(nn.ReLU())
        classifier_modules.append(nn.Dropout(p=config['dropout_2']))
        classifier_modules.append(nn.Linear(config['dense_neurons'], num_classes))

        self.backbone.classifier = nn.Sequential(*classifier_modules)

    def forward(self, x):
        return self.backbone(x)

# =====================================================================
# CELL 5: VALIDATION-LOSS CHECKPOINTED TRAINING ENGINE
# =====================================================================
def train_and_evaluate_model(config, train_loader, val_loader, device, epochs=15):
    model = TunableMonkeyEfficientNet(num_classes=10, config=config).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=config['learning_rate'], weight_decay=1e-3)

    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []

    best_val_loss = float('inf')
    best_val_acc = 0.0

    print(f"\n{'Epoch':<6} | {'Train Loss':<10} | {'Train Acc':<10} | {'Val Loss':<10} | {'Val Acc':<10}")
    print("-" * 65)

    for epoch in range(epochs):
        # PHASE 1: TRAINING STEP
        model.train()
        running_train_loss, train_correct, train_total = 0.0, 0, 0
        for imgs, targets in train_loader:
            imgs, targets = imgs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_train_loss += loss.item() * imgs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            train_total += targets.size(0)
            train_correct += (predicted == targets).sum().item()

        epoch_train_loss = running_train_loss / train_total
        epoch_train_acc = 100 * train_correct / train_total
        train_losses.append(epoch_train_loss)
        train_accuracies.append(epoch_train_acc)

        # PHASE 2: VALIDATION STEP
        model.eval()
        running_val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, targets in val_loader:
                imgs, targets = imgs.to(device), targets.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, targets)

                running_val_loss += loss.item() * imgs.size(0)
                _, predicted = torch.max(outputs.data, 1)
                val_total += targets.size(0)
                val_correct += (predicted == targets).sum().item()

        epoch_val_loss = running_val_loss / val_total
        epoch_val_acc = 100 * val_correct / val_total
        val_losses.append(epoch_val_loss)
        val_accuracies.append(epoch_val_acc)

        # Automated Checkpoint logic saving directly to your /Models directory structure
        checkpoint_msg = ""
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_val_acc = epoch_val_acc
            torch.save(model.state_dict(), 'Models/best_efficientnet_monkey.pth')
            checkpoint_msg = " 🌟 [Saved Best Weights]"

        print(f"#{epoch+1:<5} | {epoch_train_loss:<10.4f} | {epoch_train_acc:<9.2f}% | {epoch_val_loss:<10.4f} | {epoch_val_acc:<9.2f}%{checkpoint_msg}")

    print(f"\n🏆 Training completed! Absolute best validation accuracy captured: {best_val_acc:.2f}%")
    return {
        'train_losses': train_losses, 'val_losses': val_losses,
        'train_accuracies': train_accuracies, 'val_accuracies': val_accuracies
    }

# =====================================================================
# CELL 6: EXECUTE OPTIMIZED TRAINING SEQUENCE
# =====================================================================
advanced_config = {
    'dense_neurons': 512,
    'dropout_1': 0.4,
    'dropout_2': 0.3,
    'use_batchnorm': True,
    'learning_rate': 5e-4
}

history = train_and_evaluate_model(
    config=advanced_config,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=15
)

# =====================================================================
# CELL 7: PLOT HISTORICAL PERFORMANCE MAPS
# =====================================================================
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history['train_accuracies'], label='Train Accuracy', color='blue', marker='o')
plt.plot(history['val_accuracies'], label='Val Accuracy', color='orange', marker='s')
plt.title('Step-by-Step Accuracy Progress')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_losses'], label='Train Loss', color='blue', marker='o')
plt.plot(history['val_losses'], label='Val Loss', color='orange', marker='s')
plt.title('Fluctuating Loss Optimization Trend')
plt.xlabel('Epochs')
plt.ylabel('Loss Value')
plt.grid(True)
plt.legend()

plt.tight_layout()

# Save performance charts directly into your /Outputs subfolder tree
plt.savefig('Outputs/model3_training_metrics.png', dpi=300, bbox_inches='tight')
print("📁 Target check: Evaluation metrics exported seamlessly to /Outputs directory.")
plt.show()

# =====================================================================
# CELL 8: VERIFY WEIGHT PRODUCTION ARTIFACTS
# =====================================================================
target_weights = 'Models/best_efficientnet_monkey.pth'

if os.path.exists(target_weights):
    file_size_mb = os.path.getsize(target_weights) / (1024 * 1024)
    print(f"✅ CONFIRMED: '{target_weights}' successfully packed in directory structure!")
    print(f"📦 Payload Profile: {file_size_mb:.2f} MB parameters verified.")
else:
    print(f"❌ CRITICAL EXCEPTION: '{target_weights}' missing from production tracks.")

# =====================================================================
# CELL 9: PRODUCTION RE-ENTRY AND VERIFICATION INFRASTRUCTURE
# =====================================================================
production_model = TunableMonkeyEfficientNet(num_classes=10, config=advanced_config)
production_model.load_state_dict(torch.load(target_weights, map_location='cpu'))
production_model.eval()

print("\n🏆 Production-grade validation complete. Weights successfully anchored and standing by for deployment!")